In [0]:
# P17: working agent for customer-support
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage,AIMessage, HumanMessage
from langchain_core.messages.tool import ToolMessage
from langgraph.graph import StateGraph

#1: Define our single business tool
@tool
def cancel_order(order_id:str) -> str:
    """Cancel an order that hasn't shipped."""
    #(Here you'd call your real backend API)
    return f"Order {order_id} has been cancelled."
#2: The agent "brain": invoke LLm, run tool, then invoke LLM again
def call_model(state):
    msgs = state["messages"]
    order = state.get("order", {"order_id":"UNKNOWN"})

    #sytem prompt tells the model exactly what to do
    prompt = (
        f'''You are an ecommerce support agent.
        ORDER ID: {order['order_id']}
        If the customer asks to cancel, call cancel_order(order_id)
        and then send a simple confirmation.
        Otherwise, just respond normally.'''
    )
    full = [SystemMessage(prompt)] + msgs
    #1st LLM pass: decides wether to call our tool
    AIMessage = ChatOllama(model="gemma3:1b",temperature=0)(full)
    out=[first]

    if getattr(first, "tool_calls", None):
        #run the cancel_order tool
        tc = first.tool_calls[0]
        result =cancel_order(**tc["args"])
        out.append(ToolMessage(content=result, tool_call_id=tc["id"]))
        #2nd LLM pass: generate the final confirmation text
        AIMessage = ChatOllama(model="gemma3:1b",temperature=0)(full + out)
        out.appned(second)
    return {"messages": out}
#3: Write it all up in a StateGraph
def construct_graph():
    graph = StateGraph({"order": None, "messages": []})
    graph.add_node("assistant", call_model)
    graph.set_entry_point("assistant")
    return graph.compile()

graph = construct_graph()

if __name__ == "__main__":
    example_order = {"order_id": "A12345"}
    convo = [HumanMessage(content="Hi, I want to cancel my order A12345")]
    result=graph.invoke({"order": example_order, "messages": convo})
    for msg in result["messages"]:
        print(f"{msg.type}: {msg.content}")